# 02 · Rerun the May 2026 benchmark grid

Re-executes the grid behind `results_rigorous_20260523_115442.csv` with the May 2026 measurement
code (`bench/baseline_runners.py`, unchanged except paths) — **only if you choose to**.

| `MODE` | what happens | time |
|---|---|---|
| `"replay"` (default) | nothing is mined; shows the attempted May 2026 grid and its outcomes | seconds |
| `"smoke"` | Groceries + T12.D5k, k=2,3, support 0.01, every method, 1 warm-up + 3 runs | ~2 min |
| `"run"` | the full May 2026 grid (4,955 cells incl. the 64 that failed in May), resumable | ~100–120 h on 8 vCPUs |

Protocol (as recorded in the May raw files): every run is a fresh process under
`ulimit -v 60 GiB` + `timeout 1500` + `/usr/bin/time -v`; 1 warm-up probe + 3 measured runs
(`RUNS_BY_SIZE_CLASS = 3/3/3`); a failed probe ends the cell; `fastapriori_fast_1t` runs with
`RAYON_NUM_THREADS=1`, `fast`/`classic` with Rayon's default; Python methods report the library-call
time, Borgelt the process wall time.

A run writes **only** to `results/runs/<CAMPAIGN>/raw/results_rigorous_raw_<CAMPAIGN>.csv` plus a
`manifest.json`; then `FA_CAMPAIGN=<CAMPAIGN>` makes notebooks 03–07 analyse it. Restarting the
notebook with the same `CAMPAIGN` resumes where it stopped. Linux only.

In [1]:
# ---- bundle configuration (same cell in every notebook) ----
import os, sys, json, time, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

BUNDLE = Path.cwd().resolve().parent          # notebooks run from <bundle>/notebooks
assert (BUNDLE / "bench" / "baseline_runners.py").exists(), f"start Jupyter in <bundle>/notebooks (cwd={Path.cwd()})"
sys.path.insert(0, str(BUNDLE / "bench"))
DATA = Path(os.environ.get("FA_DATA_DIR", BUNDLE / "data"))
RESULTS = BUNDLE / "results"
DERIVED = RESULTS / "derived"                       # regenerated statistics, tables and figures

MAY = "campaign_2026-05"                           # the April/May 2026 GCE campaign
MAY_DIR = RESULTS / MAY
CAMPAIGN = os.environ.get("FA_CAMPAIGN", MAY) # set FA_CAMPAIGN=<id> to analyse a new run
CAMPAIGN_DIR = MAY_DIR if CAMPAIGN == MAY else RESULTS / "runs" / CAMPAIGN
AGG_CSV = (MAY_DIR / "results_rigorous_20260523_115442.csv" if CAMPAIGN == MAY
           else CAMPAIGN_DIR / f"results_rigorous_{CAMPAIGN}.csv")
LCM_CSV = MAY_DIR / "lcm_declat_results_20260523_055523.csv"
COST_CSV = (DERIVED / "cost_stats.csv") if (DERIVED / "cost_stats.csv").exists() else MAY_DIR / "cost_stats_20261022.csv"

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def rel(p):
    try:
        return Path(p).resolve().relative_to(BUNDLE).as_posix()
    except ValueError:
        return str(p)

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print(f"campaign : {CAMPAIGN}\nagg csv  : {rel(AGG_CSV)}")

campaign : campaign_2026-05
agg csv  : results/campaign_2026-05/results_rigorous_20260523_115442.csv


In [2]:
MODE = os.environ.get("FA_MODE", "replay")   # "replay" | "smoke" | "run"
RUN_CAMPAIGN = os.environ.get("FA_RUN_CAMPAIGN", time.strftime("rerun_%Y%m%d"))
# optional filters for MODE="run" (None = everything in the May 2026 grid)
ONLY_DATASETS = None      # e.g. ["Groceries", "Kosarak"]
ONLY_METHODS = None       # e.g. ["fastapriori_fast", "pyfim_eclat"]
ONLY_K = None             # e.g. [2]
TIMEOUT_S, MEMOUT_GB = 1500, 60
grid = pd.read_csv(MAY_DIR / "grid_2026-05.csv")
print(f"May 2026 grid: {len(grid)} cells;", grid.may_probe_status.value_counts().to_dict())
grid.pivot_table(index="dataset", columns="method", values="k", aggfunc="count", fill_value=0)

May 2026 grid: 4955 cells; {'ok': 4891, 'oom': 42, 'timeout': 22}


method,borgelt_apriori,borgelt_eclat,borgelt_fpgrowth,efficient_apriori,fastapriori_classic,fastapriori_fast,fastapriori_fast_1t,pyfim_apriori,pyfim_eclat,pyfim_fpgrowth
dataset,,,,,,,,,,
BMS-WebView-1,31,31,31,5,31,33,5,31,31,31
BMS-WebView-2,39,39,39,5,39,40,5,39,39,39
Chainstore,39,39,39,5,39,40,5,39,39,39
Groceries,39,39,39,5,39,40,5,39,39,39
Instacart,39,39,39,5,39,40,5,39,39,39
Kosarak,31,31,31,5,31,33,5,31,31,31
Online Retail,31,31,31,5,32,33,5,31,31,31
Retail (Belgian),39,39,39,5,39,40,5,39,39,39
T10.I4.D100k.N25k.L25k,32,32,32,0,29,33,5,32,32,32


In [3]:
if MODE == "smoke":
    RUN_CAMPAIGN = "smoke_" + time.strftime("%Y%m%d_%H%M%S")
    methods = sorted(grid.method.unique())
    tasks = pd.DataFrame([(d, m, k, 0.01) for d in ["Groceries", "T12.I4.D5k.N1k.L1k"]
                          for m in methods for k in (2, 3)
                          if not (m in ("fastapriori_fast_1t", "efficient_apriori") and k != 2)
                          and not (m == "efficient_apriori" and d.startswith("T"))],
                         columns=["dataset", "method", "k", "min_support"])
    TIMEOUT_S = 300
elif MODE == "run":
    tasks = grid.copy()
    if ONLY_DATASETS: tasks = tasks[tasks.dataset.isin(ONLY_DATASETS)]
    if ONLY_METHODS:  tasks = tasks[tasks.method.isin(ONLY_METHODS)]
    if ONLY_K:        tasks = tasks[tasks.k.isin(ONLY_K)]
    # same order as the May 2026 runs: dataset, then k, then support, then method
    tasks = tasks.sort_values(["dataset", "k", "min_support", "method"], ascending=[True, True, False, True])
else:
    tasks = grid.iloc[0:0]
print(MODE, "->", len(tasks), "cells", ("" if MODE == "replay" else f"campaign {RUN_CAMPAIGN}"))

replay -> 0 cells 


## Run (skipped in replay mode)

In [4]:
if len(tasks):
    import platform, subprocess, importlib.metadata as md
    assert sys.platform.startswith("linux"), "the measurement protocol needs Linux (/usr/bin/time, ulimit)"
    import baseline_runners as br
    from load_datasets import ALL_DATASETS
    br.RUNS_BY_SIZE_CLASS = {"fast": 3, "medium": 3, "heavy": 3}     # as in the May campaign
    run_dir = RESULTS / "runs" / RUN_CAMPAIGN
    (run_dir / "raw").mkdir(parents=True, exist_ok=True)
    br.RAW_CSV = run_dir / "raw" / f"results_rigorous_raw_{RUN_CAMPAIGN}.csv"
    env = br.collect_environment()
    import fastapriori._fastapriori_rs as rs
    br.WORKDIR.mkdir(parents=True, exist_ok=True)
    workdir_fs = subprocess.run(["stat", "-f", "-c", "%T", str(br.WORKDIR)], capture_output=True, text=True).stdout.strip()
    if workdir_fs in ("nfs", "cifs", "smb2", "fuseblk", "fuse", "9p", "v9fs", "drvfs"):
        print(f"WARNING: work dir {br.WORKDIR} is on '{workdir_fs}'. Borgelt reads its input from there inside "
              "the timed region; set FA_WORKDIR to local disk for comparable timings.")
    manifest = run_dir / "manifest.json"
    if manifest.exists():      # never resume on another build: medians would mix builds
        started_on = json.loads(manifest.read_text())["extension_sha256"]
        assert started_on == sha256(Path(rs.__file__)), (
            f"{RUN_CAMPAIGN} was started on another build; use a new FA_RUN_CAMPAIGN")
    else:
        manifest.write_text(json.dumps(dict(
            campaign=RUN_CAMPAIGN, mode=MODE, started_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            fastapriori=md.version("fastapriori"), extension=rs.__file__,
            extension_sha256=sha256(Path(rs.__file__)), workdir=str(br.WORKDIR), workdir_fs=workdir_fs,
            pyfim=md.version("pyfim"),
            efficient_apriori=md.version("efficient-apriori"),
            tools_sha256={p.name: sha256(p) for p in sorted(br.TOOLS_BIN.glob("*")) if p.is_file()},
            harness_sha256={p.name: sha256(p) for p in sorted((BUNDLE / "bench").glob("*.py"))},
            timeout_s=TIMEOUT_S, memout_gb=MEMOUT_GB, runs_by_size_class=br.RUNS_BY_SIZE_CLASS,
            n_cells=len(tasks), environment=env), indent=1, default=str))
    raw_so_far = br.load_raw()
    frames, t_start = {}, time.monotonic()
    for i, (dataset, method, k, support) in enumerate(tasks[["dataset", "method", "k", "min_support"]].itertuples(index=False)):
        existing = br.already_done(raw_so_far, dataset, method, int(k), float(support))
        if len(existing) >= 4 or (0 in existing and len(existing) == 1 and
                                  raw_so_far.loc[(raw_so_far.dataset == dataset) & (raw_so_far.method == method)
                                                 & (raw_so_far.k == k) & np.isclose(raw_so_far.min_support, support)
                                                 & (raw_so_far.run_id == 0), "status"].ne("ok").any()):
            continue   # complete, or the probe already failed (a failed probe ends the cell)
        if dataset not in frames:
            frames = {dataset: ALL_DATASETS[dataset]()}      # keep one dataset in memory
        rows = br.measure(method, dataset, frames[dataset], int(k), float(support),
                          timeout_s=TIMEOUT_S, memout_kb=MEMOUT_GB * 1024 * 1024,
                          env_snapshot=env, existing_run_ids=existing)
        for row in rows:
            br.append_raw_row(row)
        raw_so_far = br.load_raw()
        st = [r.get("status") for r in rows]
        print(f"[{i + 1}/{len(tasks)}] {dataset} | {method} | k={k} s={support:g}: {st} "
              f"({(time.monotonic() - t_start) / 60:.1f} min)", flush=True)
    print("done; aggregate with: FA_CAMPAIGN=" + RUN_CAMPAIGN + " -> 03_aggregate.ipynb")

## Smoke check: itemset counts must agree across methods

In [5]:
if MODE == "smoke":
    raw = br.load_raw()
    ok = raw[(raw.status == "ok") & (raw.run_id > 0)]
    counts = ok.pivot_table(index=["dataset", "k"], columns="method", values="n_itemsets", aggfunc="first")
    display(raw.groupby(["method", "status"]).size().unstack(fill_value=0))
    display(counts)
    disagree = counts[counts.nunique(axis=1) > 1]
    assert (raw.status == "ok").all(), raw[raw.status != "ok"][["dataset", "method", "k", "status", "stderr_tail"]]
    assert disagree.empty, disagree
    print("SMOKE OK:", len(ok), "measured runs, counts agree")